# Predicting Adult ICU In-Hospital Mortality with MIMIC-III

by: `JC Diamante`

This tutorial teaches a reproducible tabular machine learning workflow using protected MIMIC-III v1.4 data. The goal is not to deploy a clinical tool. The goal is to learn how to define a cohort, avoid outcome leakage, engineer first-day features, evaluate imbalanced classifiers, and explain model behavior.

## Protected Data Note

MIMIC-III is a credentialed dataset. Do not upload raw CSV files, patient-level excerpts, or generated row-level outputs to public GitHub or public Colab. This notebook assumes the CSV files remain in the local course folder.

In [1]:
# Import the reusable tutorial pipeline.
from mimic_mortality_tutorial import run_full_analysis, ARTIFACT_DIR

# The first run scans LABEVENTS.csv.gz and caches derived features.
# Later runs reuse mimic_mortality_artifacts/adult_icu_first24_features.csv.
results = run_full_analysis(force_rebuild=False)
results.summary

{
  "cohort_rows": 49695,
  "mortality_count": 5739,
  "mortality_rate": 0.11548445517657711,
  "feature_count": 54,
  "numeric_feature_count": 49,
  "categorical_feature_count": 5,
  "test_rows": 9939,
  "metrics": [
    {
      "model": "Logistic regression",
      "auroc": 0.8143963655417303,
      "average_precision": 0.4141640115995059,
      "balanced_accuracy": 0.7336325419131143,
      "true_negatives": 6428,
      "false_positives": 2363,
      "false_negatives": 303,
      "true_positives": 845
    },
    {
      "model": "Random forest",
      "auroc": 0.8423966227734495,
      "average_precision": 0.4717287085118528,
      "balanced_accuracy": 0.7383819153814659,
      "true_negatives": 7507,
      "false_positives": 1284,
      "false_negatives": 433,
      "true_positives": 715
    }
  ],
  "lab_itemids": {
    "50882": "bicarbonate",
    "50902": "chloride",
    "50912": "creatinine",
    "50931": "glucose",
    "50971": "potassium",
    "50983": "sodium",
    "51006": "

## 1. Cohort Definition

We create one row per adult hospital admission with a first ICU stay. We exclude neonatal ICU stays and patients under 18. The label is `HOSPITAL_EXPIRE_FLAG`, renamed to `MORTALITY_LABEL`.

In [2]:
results.features[['AGE', 'GENDER', 'FIRST_CAREUNIT', 'MORTALITY_LABEL']].head(8)

AGE,GENDER,FIRST_CAREUNIT,MORTALITY_LABEL
35.475110,F,MICU,0
59.910963,M,MICU,0
48.915514,F,MICU,0
73.822288,F,SICU,0
60.798046,M,CSRU,0
54.520017,F,SICU,0
21.503551,M,TSICU,0
67.716747,M,CSRU,0


## 2. Leakage Control

The model only uses information available during the first 24 hours after ICU admission. ICD diagnosis codes are intentionally excluded because they are usually finalized after the stay and can leak outcome information.

In [3]:
# The feature table contains demographics, admission context, ICU unit, and first-24h lab summaries.
results.features.head(8)

AGE,GENDER,ADMISSION_TYPE,INSURANCE,ETHNICITY_GROUP,FIRST_CAREUNIT,MORTALITY_LABEL,bicarbonate_min,bun_min,chloride_min,creatinine_min,glucose_min,hematocrit_min,hemoglobin_min,lactate_min,platelets_min,potassium_min,sodium_min,wbc_min,bicarbonate_max,bun_max,chloride_max,creatinine_max,glucose_max,hematocrit_max,hemoglobin_max,lactate_max,platelets_max,potassium_max,sodium_max,wbc_max,bicarbonate_mean,bun_mean,chloride_mean,creatinine_mean,glucose_mean,hematocrit_mean,hemoglobin_mean,lactate_mean,platelets_mean,potassium_mean,sodium_mean,wbc_mean,bicarbonate_count,bun_count,chloride_count,creatinine_count,glucose_count,hematocrit_count,hemoglobin_count,lactate_count,platelets_count,potassium_count,sodium_count,wbc_count
35.475110,F,EMERGENCY,Private,WHITE,MICU,0,17.0,33.0,110.0,2.2,123.0,32.2,11.0,NaN,376.0,3.8,141.0,10.4,21.0,42.0,114.0,2.4,185.0,32.8,11.3,NaN,407.0,4.2,144.0,11.2,18.750000,37.750000,111.50,2.300000,151.000000,32.500000,11.150000,NaN,391.500000,4.075,142.50,10.800000,4.0,4.0,4.0,4.0,4.0,2.0,2.0,NaN,2.0,4.0,4.0,2.0
59.910963,M,EMERGENCY,Private,WHITE,MICU,0,15.0,46.0,111.0,1.1,56.0,21.3,7.1,1.1,148.0,5.0,133.0,13.4,16.0,49.0,112.0,1.2,113.0,31.8,10.4,1.1,162.0,5.0,133.0,14.2,15.500000,47.500000,111.50,1.150000,84.500000,27.900000,9.166667,1.1,157.333333,5.000,133.00,13.933333,2.0,2.0,2.0,2.0,2.0,5.0,3.0,1.0,3.0,2.0,2.0,3.0
48.915514,F,EMERGENCY,Private,BLACK,MICU,0,24.0,16.0,103.0,0.6,129.0,30.6,10.3,NaN,204.0,3.7,131.0,13.6,24.0,16.0,103.0,0.6,129.0,30.6,10.3,NaN,204.0,3.7,131.0,13.6,24.000000,16.000000,103.00,0.600000,129.000000,30.600000,10.300000,NaN,204.000000,3.700,131.00,13.600000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0
73.822288,F,EMERGENCY,Private,WHITE,SICU,0,25.0,12.0,106.0,0.6,115.0,33.5,11.3,NaN,198.0,4.1,138.0,8.8,27.0,14.0,107.0,0.7,137.0,37.1,12.3,NaN,259.0,4.7,140.0,12.3,26.000000,13.000000,106.50,0.650000,126.000000,35.300000,11.800000,NaN,228.500000,4.400,139.00,10.550000,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,2.0,2.0,2.0,2.0
60.798046,M,EMERGENCY,Private,WHITE,CSRU,0,23.0,12.0,109.0,0.8,177.0,35.8,12.1,1.1,125.0,4.0,140.0,12.3,24.0,13.0,111.0,0.8,177.0,39.9,13.7,1.5,149.0,4.4,143.0,17.3,23.500000,12.500000,110.00,0.800000,177.000000,37.600000,12.966667,1.3,139.333333,4.200,141.50,14.466667,2.0,2.0,2.0,2.0,1.0,3.0,3.0,2.0,3.0,2.0,2.0,3.0
54.520017,F,ELECTIVE,Private,WHITE,SICU,0,26.0,12.0,107.0,1.2,133.0,25.0,9.7,NaN,347.0,4.0,138.0,15.0,26.0,12.0,107.0,1.2,133.0,29.1,9.7,NaN,347.0,4.0,138.0,15.0,26.000000,12.000000,107.00,1.200000,133.000000,26.533333,9.700000,NaN,347.000000,4.000,138.00,15.000000,1.0,1.0,1.0,1.0,1.0,3.0,1.0,NaN,1.0,1.0,1.0,1.0
21.503551,M,EMERGENCY,Medicaid,HISPANIC/LATINO,TSICU,0,23.0,11.0,109.0,1.1,113.0,25.0,9.0,2.3,188.0,3.8,137.0,7.6,24.0,12.0,112.0,1.3,160.0,37.9,13.2,2.3,295.0,4.5,141.0,17.3,23.666667,11.666667,110.00,1.200000,135.333333,29.175000,10.566667,2.3,228.000000,4.100,138.75,11.566667,3.0,3.0,4.0,3.0,3.0,4.0,3.0,1.0,3.0,4.0,4.0,3.0
67.716747,M,EMERGENCY,Medicare,WHITE,CSRU,0,23.0,10.0,100.0,0.7,96.0,22.2,8.2,2.1,138.0,4.1,129.0,7.0,26.0,14.0,104.0,0.9,96.0,29.5,11.0,2.5,200.0,4.6,133.0,14.1,24.333333,12.000000,102.25,0.766667,96.000000,25.700000,9.380000,2.3,175.400000,4.450,131.75,10.580000,3.0,3.0,4.0,3.0,1.0,7.0,5.0,2.0,5.0,4.0,4.0,5.0


## 3. Model Training

The pipeline uses a stratified train/test split. Numeric features are median-imputed and scaled. Categorical features are imputed and one-hot encoded. We compare logistic regression and random forest baselines.

In [4]:
results.metrics.round(4)

model,auroc,average_precision,balanced_accuracy,true_negatives,false_positives,false_negatives,true_positives
Logistic regression,0.8144,0.4142,0.7336,6428,2363,303,845
Random forest,0.8424,0.4717,0.7384,7507,1284,433,715


## 4. Evaluation

Because mortality is an imbalanced outcome, accuracy alone is misleading. The tutorial reports AUROC, average precision, balanced accuracy, and a confusion matrix.

![ROC curve](mimic_mortality_artifacts/roc_curve.png)

![Precision-recall curve](mimic_mortality_artifacts/precision_recall_curve.png)

![Confusion matrix](mimic_mortality_artifacts/confusion_matrix.png)

## 5. Interpretability

Logistic regression coefficients show directional associations after preprocessing. Permutation importance estimates which original features most affect random forest average precision.

In [5]:
results.logistic_top_features[['feature', 'coefficient']].round(4)

feature,coefficient
Creatinine Mean,-3.7723
Bun Mean,2.4287
Creatinine Max,1.8710
Bun Max,-1.8538
Creatinine Min,1.6791
Bicarbonate Mean,-1.2936
Hematocrit Mean,-0.8733
Admission Type Elective,-0.8605
Chloride Mean,-0.8368
Hematocrit Max,0.7900


In [6]:
results.rf_top_features[['feature', 'importance_mean', 'importance_std']].round(4)

feature,importance_mean
Lactate Mean,0.0312
Lactate Min,0.0304
Age,0.0301
Lactate Max,0.0222
Admission Type,0.0190
Bun Mean,0.0073
Bicarbonate Min,0.0062
First Careunit,0.0054
Bicarbonate Mean,0.0034
Bicarbonate Max,0.0034


![Logistic coefficients](mimic_mortality_artifacts/logistic_coefficients.png)

![Random forest permutation importance](mimic_mortality_artifacts/rf_permutation_importance.png)

## 6. Limitations

- This is an educational model, not a validated bedside decision system.
- Missingness can encode care patterns as well as physiology.
- MIMIC dates are shifted and identities are protected.
- External validation would be required before any clinical claim.